# PINN experiments on Colab
Switch the runtime to a GPU first (Runtime > Change runtime type). A free GPU is not guaranteed and sessions can be cut off.
Runs are checkpointed and finished runs are skipped, so after a disconnect just run the cells again.
Keeping the repo on Google Drive means the results and checkpoints survive.

In [ ]:
CASE = 'venturi'          # 'venturi' or 'cavity'
QUICK = False             # True: short test run (results go to the *_reduced files)
USE_DRIVE = True
REPO_URL = 'https://github.com/vinothkannac/pinn-flow-reconstruction.git'

In [ ]:
import os, subprocess
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/pinn-flow-reconstruction'
else:
    WORK = '/content/pinn-flow-reconstruction'
if not os.path.isdir(WORK):
    subprocess.run(['git', 'clone', REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], check=False)
os.chdir(WORK)
print(os.getcwd())

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert os.path.exists(f'data/{CASE}/sensors'), 'sensor files are missing in the repo'

## Run the experiments

In [ ]:
cmd = ['python', 'experiments/run_experiments.py', '--case', CASE, '--device', 'auto']
if QUICK:
    cmd.append('--quick')
print(' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('exit code', proc.returncode)

## Tables and figures

In [ ]:
flag = '--reduced' if QUICK else ''
sub = 'reduced' if QUICK else 'full'
!python experiments/summarize_results.py --case {CASE} {flag}
!python scripts/reporting/make_media.py --case {CASE} --what comparison walls loss video --sub {sub}

In [ ]:
import pandas as pd
pd.read_csv(f"results/{CASE}/" + ('metrics_reduced.csv' if QUICK else 'metrics.csv')).tail(10)